In [30]:
import json
import re
from huggingface_hub import InferenceClient
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
client = InferenceClient(api_key=hf_token)

prompt = """
Extraia as informações da seguinte fatura e retorne APENAS um JSON válido.
Chaves esperadas:
- "invoice_number" (inteiro)
- "issue_date" (string no formato AAAA-MM-DD)
- "grand_total" (número flutuante)
- "is_paid" (booleano)

Texto da fatura: Fatura #INV-5501 emitida em 15 de Março de 2026. Total a pagar: $ 1,450.50. Status: Aguardando pagamento.
"""

try:
    response = client.chat.completions.create(
        model="Qwen/Qwen2.5-7B-Instruct",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=200
    )
    raw_output = response.choices[0].message.content
    print("Raw output:", raw_output)

    # Extração robusta de JSON
    json_match = re.search(r'\{.*\}', raw_output, re.DOTALL)
    if json_match:
        clean_json = json_match.group(0)
        data = json.loads(clean_json)
        print("Parsed Data:", data)
    else:
        print("JSON não encontrado na resposta.")

except Exception as e:
    print(f"Erro na execução: {e}")

Raw output: ```json
{
  "invoice_number": 5501,
  "issue_date": "2026-03-15",
  "grand_total": 1450.50,
  "is_paid": false
}
```
Parsed Data: {'invoice_number': 5501, 'issue_date': '2026-03-15', 'grand_total': 1450.5, 'is_paid': False}


In [36]:
!pip install "pydantic[email]" -q

In [35]:
from pydantic import BaseModel, Field, model_validator, EmailStr
from typing import List, Optional, Literal
from datetime import date
from decimal import Decimal

class LineItem(BaseModel):
    description: str = Field(..., description="Nome claro do produto ou serviço prestado")
    quantity: float = Field(default=1.0, description="A quantidade adquirida")
    unit_price: Decimal = Field(..., description="O preço por unidade")
    total_line_price: Decimal = Field(..., description="quantity * unit_price")

class InvoiceExtraction(BaseModel):
    extraction_status: Literal["success", "partial", "failed"]
    invoice_id: Optional[str] = None
    issue_date: date
    vendor_name: str
    vendor_email: Optional[EmailStr] = None
    items: List[LineItem] = Field(default_factory=list)
    tax_amount: Decimal = Field(default=Decimal('0.0'))
    grand_total: Decimal
    confidence_rationale: str

    @model_validator(mode='after')
    def verify_financial_integrity(self) -> 'InvoiceExtraction':
        calculated_total = sum((item.total_line_price for item in self.items), Decimal('0'))
        if abs(calculated_total + self.tax_amount - self.grand_total) > Decimal('0.5'):
            raise ValueError(f"Inconsistência financeira: Soma {calculated_total} + Taxas {self.tax_amount} != Total {self.grand_total}")
        return self

print("Schema Pydantic corrigido e dependências instaladas.")

Schema Pydantic corrigido e dependências instaladas.


In [40]:
from pydantic import BaseModel, Field, ValidationError, BeforeValidator
from datetime import date
from typing import Optional, Annotated

# Função auxiliar para garantir que invoice_number vire string
def force_string(v):
    return str(v)

# Definindo o contrato de dados de forma declarativa
class InvoiceSchema(BaseModel):
    # Usando Annotated com BeforeValidator para forçar a conversão de int para string antes da validação
    invoice_number: Annotated[str, BeforeValidator(force_string)] = Field(description="O ID único ou número da fatura")
    issue_date: date = Field(description="A data de emissão no formato AAAA-MM-DD")
    grand_total: float = Field(gt=0, description="O valor total da fatura, sem símbolos de moeda")
    is_paid: bool = Field(default=False, description="Verdadeiro se a fatura já estiver paga")
    vendor_email: Optional[str] = Field(default=None, description="Email do fornecedor, se houver")

# Simulando um retorno imperfeito do LLM
llm_imperfect_json = {
    "invoice_number": 5501,              # Retornou int em vez de str
    "issue_date": "2026-03-15",          # String em vez de datetime.date
    "grand_total": "1450.50",            # String em vez de float
    "is_paid": "true"                    # String "true" em vez de booleano True
}

try:
    # O Pydantic intercepta os dados, coage e valida em uma única linha
    validated_invoice = InvoiceSchema(**llm_imperfect_json)

    print(f"Invoice Number: {validated_invoice.invoice_number} (Tipo: {type(validated_invoice.invoice_number)})")
    print(f"Issue Date: {validated_invoice.issue_date} (Tipo: {type(validated_invoice.issue_date)})")
    print(f"Grand Total: {validated_invoice.grand_total} (Tipo: {type(validated_invoice.grand_total)})")
    print(f"is_paid: {validated_invoice.is_paid} (Tipo: {type(validated_invoice.is_paid)})")
    print("Validação concluída com sucesso!")

except ValidationError as e:
    print("O Pydantic capturou os erros detalhadamente:")
    print(e.json())

Invoice Number: 5501 (Tipo: <class 'str'>)
Issue Date: 2026-03-15 (Tipo: <class 'datetime.date'>)
Grand Total: 1450.5 (Tipo: <class 'float'>)
is_paid: True (Tipo: <class 'bool'>)
Validação concluída com sucesso!


In [42]:
import re
import json
from pydantic import BaseModel
from huggingface_hub import InferenceClient
from google.colab import userdata

class PaperAnalysis(BaseModel):
    title: str
    abstract_summary: str
    key_findings: list[str]

hf_token = userdata.get('HF_TOKEN')
client = InferenceClient(api_key=hf_token)

# Fornecendo o esquema diretamente no prompt como fallback seguro
messages = [
    {"role": "user",
     "content": f"Analise o paper sobre Attention is All You Need. Retorne APENAS um JSON seguindo este esquema: {PaperAnalysis.model_json_schema()}"}
]

try:

    response = client.chat.completions.create(
        messages=messages,
        model="Qwen/Qwen2.5-7B-Instruct",
        max_tokens=500
    )

    content = response.choices[0].message.content
    # Tenta encontrar o JSON no texto caso o modelo retorne Markdown
    match = re.search(r'\{.*\}', content, re.DOTALL)
    if match:
        data = json.loads(match.group(0))
        validated = PaperAnalysis(**data)
        print("Dados validados:", validated.model_dump())
    else:
        print("Conteúdo não estruturado:", content)
except Exception as e:
    print(f"Erro: {e}")

Dados validados: {'title': 'Attention is All You Need', 'abstract_summary': 'Este trabalho apresenta um novo modelo de arquitetura de redes neurais para processamento de linguagem natural, baseado exclusivamente em atenção, eliminando a necessidade de componentes como camadas de recurso escondido e camadas de transformação. O modelo é capaz de traduzir de uma linguagem para outra com alta precisão, utilizando apenas mecanismos de atenção.', 'key_findings': ['Introdução de um novo modelo baseado em atenção que elimina a necessidade de camadas de recurso escondido e transformação.', 'Melhoria significativa na precisão da tradução de linguagens.', 'Redução do tempo de treinamento e da complexidade do modelo.', 'Aplicabilidade em uma variedade de tarefas de processamento de linguagem natural.']}


In [43]:
from datetime import date
from decimal import Decimal
from pydantic import ValidationError

# 1. Simulação de Dados Válidos
valid_invoice_data = {
    "extraction_status": "success",
    "invoice_id": "INV-2024-001",
    "issue_date": date(2024, 5, 20),
    "vendor_name": "Tech Solutions Inc.",
    "vendor_email": "support@techsolutions.com",
    "items": [
        {
            "description": "Consultoria Cloud",
            "quantity": 2.0,
            "unit_price": Decimal("500.00"),
            "total_line_price": Decimal("1000.00")
        },
        {
            "description": "Licença Software",
            "quantity": 1.0,
            "unit_price": Decimal("250.50"),
            "total_line_price": Decimal("250.50")
        }
    ],
    "tax_amount": Decimal("50.00"),
    "grand_total": Decimal("1300.50"), # 1000 + 250.50 + 50 = 1300.50
    "confidence_rationale": "Extração clara e valores batem perfeitamente."
}

try:
    invoice = InvoiceExtraction(**valid_invoice_data)
    print("✅ Teste 1 (Válido): Sucesso!")
    display(invoice.model_dump())
except ValidationError as e:
    print("❌ Teste 1 falhou inesperadamente:", e.json())

print("-" * 30)

# 2. Simulação de Dados Inválidos (Erro de Soma)
invalid_invoice_data = valid_invoice_data.copy()
invalid_invoice_data["grand_total"] = Decimal("2000.00") # Valor errado de propósito

try:
    InvoiceExtraction(**invalid_invoice_data)
    print("❌ Teste 2 falhou: O schema deveria ter rejeitado o valor incoerente.")
except ValidationError as e:
    print("✅ Teste 2 (Inconsistência): Capturado com sucesso!")
    print("Erro retornado:", e.errors()[0]['msg'])

✅ Teste 1 (Válido): Sucesso!


{'extraction_status': 'success',
 'invoice_id': 'INV-2024-001',
 'issue_date': datetime.date(2024, 5, 20),
 'vendor_name': 'Tech Solutions Inc.',
 'vendor_email': 'support@techsolutions.com',
 'items': [{'description': 'Consultoria Cloud',
   'quantity': 2.0,
   'unit_price': Decimal('500.00'),
   'total_line_price': Decimal('1000.00')},
  {'description': 'Licença Software',
   'quantity': 1.0,
   'unit_price': Decimal('250.50'),
   'total_line_price': Decimal('250.50')}],
 'tax_amount': Decimal('50.00'),
 'grand_total': Decimal('1300.50'),
 'confidence_rationale': 'Extração clara e valores batem perfeitamente.'}

------------------------------
✅ Teste 2 (Inconsistência): Capturado com sucesso!
Erro retornado: Value error, Inconsistência financeira: Soma 1250.50 + Taxas 50.00 != Total 2000.00
